# NEST simulation

In [1]:
import nest
import nest.voltage_trace
import matplotlib.pyplot as plt


              -- N E S T --
  Copyright (C) 2004 The NEST Initiative

 Version: 3.7.0
 Built: May 24 2024 10:10:57

 This program is provided AS IS and comes with
 NO WARRANTY. See the file LICENSE for details.

 Problems or suggestions?
   Visit https://www.nest-simulator.org

 Type 'nest.help()' to find out more about NEST.



In [2]:
# -------------------------------
# 1. Reset kernel (simulation env)
# -------------------------------
nest.ResetKernel()

In [3]:
# -------------------------------
# 2. Create neuron population
# -------------------------------
# Create 100 leaky integrate-and-fire (LIF) neurons
n_neurons = 100
neurons = nest.Create("iaf_psc_alpha", n_neurons)

In [4]:
# -------------------------------
# 3. Create external Poisson input
# -------------------------------
# Each neuron receives a noisy background input
noise = nest.Create("poisson_generator", 1, {"rate": 8000.0})  # Hz

In [5]:
# -------------------------------
# 4. Add a spike detector and multimeter
# -------------------------------
spike_detector = nest.Create("spike_recorder")
voltmeter = nest.Create("multimeter", params={"record_from": ["V_m"]})

In [6]:
# -------------------------------
# 5. Connect the devices
# -------------------------------
nest.Connect(noise, neurons, syn_spec={"weight": 50.0})
nest.Connect(voltmeter, neurons[0])  # record from the first neuron
nest.Connect(neurons, spike_detector)

In [7]:
# Add some recurrent (random) connectivity
conn_dict = {"rule": "fixed_indegree", "indegree": 10}
syn_dict = {"weight": 20.0, "delay": 1.5}
nest.Connect(neurons, neurons, conn_dict, syn_dict)

In [8]:
# -------------------------------
# 6. Simulate
# -------------------------------
nest.Simulate(1000.0)  # ms


Oct 24 16:48:08 NodeManager::prepare_nodes [Info]: 
    Preparing 103 nodes for simulation.

Oct 24 16:48:08 SimulationManager::start_updating_ [Info]: 
    Number of local nodes: 103
    Simulation time (ms): 1000
    Number of OpenMP threads: 1
    Not using MPI

Oct 24 16:48:08 SimulationManager::run [Info]: 
    Simulation finished.


In [9]:
# -------------------------------
# 7. Retrieve and plot results
# -------------------------------
events = spike_detector.events
senders = events["senders"]
times = events["times"]

In [10]:
vm = voltmeter.events
vm["times"]   # time points
vm["V_m"]     # voltage values (for the recorded neuron)


array([-70.        , -69.77616185, -68.1139599 , -65.35486508,
       -62.20102002, -58.83616597, -55.64345584, -70.        ,
       -70.        , -64.98601696, -58.27316356, -70.        ,
       -70.        , -67.11169844, -60.27692948, -70.        ,
       -70.        , -69.2006818 , -61.71864704, -70.        ,
       -70.        , -70.        , -60.47450071, -70.        ,
       -70.        , -65.67334456, -55.45285743, -70.        ,
       -70.        , -60.97426538, -70.        , -70.        ,
       -67.00249205, -57.90918007, -70.        , -70.        ,
       -64.3437501 , -55.01800916, -70.        , -70.        ,
       -60.87513465, -70.        , -70.        , -66.90949361,
       -56.89412066, -70.        , -70.        , -60.88799724,
       -70.        , -70.        , -65.95952242, -56.58709499,
       -70.        , -70.        , -62.7208826 , -70.        ,
       -70.        , -70.        , -62.88904037, -57.19946458,
       -70.        , -70.        , -66.42479159, -59.83

In [12]:
import numpy as np

N = 100
T = 1000
bin_size = 1.0  # ms

# Create a matrix of zeros: [neurons × timebins]
spike_matrix = np.zeros((N, int(T/bin_size)))

# Fill it in:
for sender, t in zip(senders, times):
    t_bin = int(t // bin_size)
    if t_bin >= spike_matrix.shape[1]:
        continue  # skip spikes exactly at the end
    spike_matrix[int(sender) - 1, t_bin] += 1



In [14]:
from scipy.ndimage import gaussian_filter1d

firing_rate_matrix = gaussian_filter1d(spike_matrix, sigma=5, axis=1)


In [16]:
firing_rate_matrix.shape

(100, 1000)

In [17]:
spike_matrix.shape

(100, 1000)